In [1]:

import requests
import os

url = "https://usegalaxy.org/api/datasets/f9cad7b01a472135786cecaaba75a2ef/display?to_ext=h5"

response = requests.get(url)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Downloaded size:", len(response.content), "bytes")

if response.status_code == 200 and len(response.content) > 0:
    with open("galaxy_best_model.h5", "wb") as f:
        f.write(response.content)
    print("File saved successfully!")
else:
    print("Download unsuccessful. Galaxy may require authentication.")


Status code: 200
Content type: application/octet-stream
Downloaded size: 3727 bytes
File saved successfully!


In [2]:

import h5py
import os

model_path = "galaxy_best_model.h5"

print("File size:", os.path.getsize(model_path), "bytes")
print("Is valid HDF5:", h5py.is_hdf5(model_path))

if h5py.is_hdf5(model_path):
    with h5py.File(model_path, "r") as f:
        print("\nTop-level contents:")
        print(list(f.keys()))
        print("\nFile attributes:")
        for key, value in f.attrs.items():
            print(key, ":", value)
else:
    with open(model_path, "rb") as f:
        print("\nFirst 32 bytes:", f.read(32))


File size: 3727 bytes
Is valid HDF5: True

Top-level contents:
['model']

File attributes:


In [3]:

with h5py.File("galaxy_best_model.h5", "r") as f:
    model = f["model"]

    print("Object type:", type(model))
    print("Model attributes:", dict(model.attrs))

    print("\nContents:")
    model.visititems(
        lambda name, obj: print(
            name,
            "| Type:", type(obj).__name__,
            "| Shape:", getattr(obj, "shape", None),
            "| Dtype:", getattr(obj, "dtype", None)
        )
    )


Object type: <class 'h5py._hl.dataset.Dataset'>
Model attributes: {}

Contents:


AttributeError: 'Dataset' object has no attribute 'visititems'

In [4]:

with h5py.File("galaxy_best_model.h5", "r") as f:
    model = f["model"]

    print("Dataset shape:", model.shape)
    print("Dataset dtype:", model.dtype)
    print("\nFirst 100 bytes of stored data:")

    data = model[()]
    print(repr(data[:100]))


Dataset shape: ()
Dataset dtype: |V1679

First 100 bytes of stored data:


IndexError: too many indices for array: array is 0-dimensional, but 1 were indexed

In [5]:

with h5py.File("galaxy_best_model.h5", "r") as f:
    data = f["model"][()]
    raw = data.tobytes()

    print("Stored data type:", type(data))
    print("Stored data size:", len(raw), "bytes")
    print("First 100 bytes:", raw[:100])
    print("Text preview:", raw[:100].decode("utf-8", errors="replace"))


Stored data type: <class 'numpy.void'>
Stored data size: 1679 bytes
First 100 bytes: b'\x80\x04\x95\x00\x02\x00\x00\x00\x00\x00\x00\x8c\x1esklearn.linear_model._logistic\x94\x8c\x12LogisticRegression\x94\x93\x94)\x81\x94}\x94(\x8c\x07penalty\x94\x8c\x02l2\x94\x8c\x04dual\x94\x89\x8c\x03to'
Text preview: ��       �sklearn.linear_model._logistic��LogisticRegression���)��}�(�penalty��l2��dual���to


In [6]:

import pickle
import h5py

with h5py.File("galaxy_best_model.h5", "r") as f:
    model_bytes = f["model"][()].tobytes()

model = pickle.loads(model_bytes)

print("Model loaded successfully!")
print("Model type:", type(model))
print("Model parameters:", model.get_params())


Model loaded successfully!
Model type: <class 'numpy.ndarray'>


AttributeError: 'numpy.ndarray' object has no attribute 'get_params'

In [7]:

print("Type:", type(model))
print("Shape:", model.shape)
print("Data type:", model.dtype)
print("Contents:")
print(repr(model))


Type: <class 'numpy.ndarray'>
Shape: (21,)
Data type: object
Contents:
array(['TMB', 'Systemic_therapy_history', 'Albumin', 'NLR', 'Age',
       'CancerType1', 'CancerType2', 'CancerType3', 'CancerType4',
       'CancerType5', 'CancerType6', 'CancerType7', 'CancerType8',
       'CancerType9', 'CancerType10', 'CancerType11', 'CancerType12',
       'CancerType13', 'CancerType14', 'CancerType15', 'CancerType16'],
      dtype=object)


In [8]:

import pickle
import h5py

with h5py.File("galaxy_best_model.h5", "r") as f:
    raw_model = f["model"][()].tobytes()

loaded_object = pickle.loads(raw_model)

print("Loaded object:", type(loaded_object))
print("Shape:", getattr(loaded_object, "shape", "Not an array"))
print("Is sklearn estimator:", hasattr(loaded_object, "predict"))
print("Has coefficients:", hasattr(loaded_object, "coef_"))
print("Has feature names:", hasattr(loaded_object, "feature_names_in_"))


Loaded object: <class 'numpy.ndarray'>
Shape: (21,)
Is sklearn estimator: False
Has coefficients: False
Has feature names: False


In [9]:

import pickletools

with h5py.File("galaxy_best_model.h5", "r") as f:
    raw = f["model"][()].tobytes()

print("Pickle structure:")
pickletools.dis(raw)


Pickle structure:
    0: \x80 PROTO      4
    2: \x95 FRAME      512
   11: \x8c SHORT_BINUNICODE 'sklearn.linear_model._logistic'
   43: \x94 MEMOIZE    (as 0)
   44: \x8c SHORT_BINUNICODE 'LogisticRegression'
   64: \x94 MEMOIZE    (as 1)
   65: \x93 STACK_GLOBAL
   66: \x94 MEMOIZE    (as 2)
   67: )    EMPTY_TUPLE
   68: \x81 NEWOBJ
   69: \x94 MEMOIZE    (as 3)
   70: }    EMPTY_DICT
   71: \x94 MEMOIZE    (as 4)
   72: (    MARK
   73: \x8c     SHORT_BINUNICODE 'penalty'
   82: \x94     MEMOIZE    (as 5)
   83: \x8c     SHORT_BINUNICODE 'l2'
   87: \x94     MEMOIZE    (as 6)
   88: \x8c     SHORT_BINUNICODE 'dual'
   94: \x94     MEMOIZE    (as 7)
   95: \x89     NEWFALSE
   96: \x8c     SHORT_BINUNICODE 'tol'
  101: \x94     MEMOIZE    (as 8)
  102: G        BINFLOAT   0.0001
  111: \x8c     SHORT_BINUNICODE 'C'
  114: \x94     MEMOIZE    (as 9)
  115: G        BINFLOAT   1.0
  124: \x8c     SHORT_BINUNICODE 'fit_intercept'
  139: \x94     MEMOIZE    (as 10)
  140: \x88     NEW

ValueError: memo key 0 already defined

In [10]:

import joblib
import h5py
import io

with h5py.File("galaxy_best_model.h5", "r") as f:
    raw = f["model"][()].tobytes()

model = joblib.load(io.BytesIO(raw))

print("Loaded object:", type(model))
print("Has prediction method:", hasattr(model, "predict"))
print("Has coefficients:", hasattr(model, "coef_"))

if hasattr(model, "get_params"):
    print("Parameters:", model.get_params())


Loaded object: <class 'sklearn.linear_model._logistic.LogisticRegression'>
Has prediction method: True
Has coefficients: True
Parameters: {'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 1000, 'multi_class': 'auto', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.4.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [11]:

print("Model classes:", model.classes_)
print("Coefficient shape:", model.coef_.shape)
print("Intercept:", model.intercept_)

print("\nFeature coefficients:")

for feature, coefficient in zip(
    ['TMB', 'Systemic_therapy_history', 'Albumin',
     'NLR', 'Age'] +
    [f'CancerType{i}' for i in range(1, 17)],
    model.coef_[0]
):
    print(f"{feature}: {coefficient:.5f}")


Model classes: [0 1]
Coefficient shape: (1, 21)
Intercept: [-3.63563335]

Feature coefficients:
TMB: 0.04600
Systemic_therapy_history: -0.97778
Albumin: 0.67557
NLR: -0.04375
Age: 0.00995
CancerType1: -0.67993
CancerType2: -0.52803
CancerType3: -0.44645
CancerType4: -0.15180
CancerType5: 0.66624
CancerType6: 0.42148
CancerType7: 0.00000
CancerType8: -0.39322
CancerType9: -0.35928
CancerType10: 0.28303
CancerType11: -0.14741
CancerType12: -0.22788
CancerType13: -1.07954
CancerType14: 0.60660
CancerType15: 0.20684
CancerType16: -0.10692


In [12]:

import pandas as pd
import sklearn

print("Scikit-learn version:", sklearn.__version__)
print("Model expects these features:")
print(list(model.feature_names_in_))

url = "https://zenodo.org/records/13885908/files/Chowell_test_Response.tsv"

test_data = pd.read_csv(url, sep="\t")

print("\nTest dataset shape:", test_data.shape)
print("\nDataset columns:")
print(list(test_data.columns))

print("\nFirst five rows:")
display(test_data.head())


Scikit-learn version: 1.6.1
Model expects these features:
['TMB', 'Systemic_therapy_history', 'Albumin', 'NLR', 'Age', 'CancerType1', 'CancerType2', 'CancerType3', 'CancerType4', 'CancerType5', 'CancerType6', 'CancerType7', 'CancerType8', 'CancerType9', 'CancerType10', 'CancerType11', 'CancerType12', 'CancerType13', 'CancerType14', 'CancerType15', 'CancerType16']

Test dataset shape: (515, 22)

Dataset columns:
['TMB', 'Systemic_therapy_history', 'Albumin', 'NLR', 'Age', 'CancerType1', 'CancerType2', 'CancerType3', 'CancerType4', 'CancerType5', 'CancerType6', 'CancerType7', 'CancerType8', 'CancerType9', 'CancerType10', 'CancerType11', 'CancerType12', 'CancerType13', 'CancerType14', 'CancerType15', 'CancerType16', 'Response']

First five rows:


,TMB,Systemic_therapy_history,Albumin,NLR,Age,CancerType1,CancerType2,CancerType3,CancerType4,CancerType5,...,CancerType8,CancerType9,CancerType10,CancerType11,CancerType12,CancerType13,CancerType14,CancerType15,CancerType16,Response
0,32.5,1,4.3,1.19,67.915127,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,1
1,19.3,1,3.8,1.38,62.502396,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
2,10.5,1,4.2,2.54,52.106776,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
3,3.5,1,3.7,4.89,61.987680,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,32.5,0,4.6,4.00,85.000000,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,1


In [13]:

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    confusion_matrix
)

# Separate input features from the actual outcomes
X_test = test_data[list(model.feature_names_in_)]
y_test = test_data["Response"]

# Get the model's predicted probabilities
probabilities = model.predict_proba(X_test)[:, 1]

# Galaxy's optimised decision threshold
threshold = 0.269
predictions = (probabilities >= threshold).astype(int)

print("MODEL VALIDATION RESULTS")
print("------------------------")
print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, probabilities), 3))
print("Precision:", round(precision_score(y_test, predictions), 3))
print("Recall:", round(recall_score(y_test, predictions), 3))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, predictions))


MODEL VALIDATION RESULTS
------------------------
Accuracy: 0.697
ROC-AUC: 0.76
Precision: 0.427
Recall: 0.669

Confusion matrix:
[[274 114]
 [ 42  85]]


In [14]:

!pip install gradio


In [21]:
import gradio as gr
import pandas as pd

# --------------------------------------------------
# 1. CANCER TYPE MAPPING
# --------------------------------------------------

cancer_mapping = {
    "Bladder": 1,
    "Breast": 2,
    "Colorectal": 3,
    "Endometrial": 4,
    "Esophageal": 5,
    "Gastric": 6,
    "Head & Neck": 7,
    "Hepatobiliary": 8,
    "Melanoma": 9,
    "Mesothelioma": 10,
    "Non-small cell lung cancer (NSCLC)": 11,
    "Ovarian": 12,
    "Pancreatic": 13,
    "Renal": 14,
    "Sarcoma": 15,
    "Small cell lung cancer (SCLC)": 16
}


# --------------------------------------------------
# 2. PREDICTION FUNCTION
# --------------------------------------------------

def predict_response(tmb, therapy, albumin, nlr, age, cancer_type):

    if None in [tmb, therapy, albumin, nlr, age, cancer_type]:
        return "<p>Please complete all the input fields.</p>"

    values = {
        "TMB": tmb,
        "Systemic_therapy_history": therapy,
        "Albumin": albumin,
        "NLR": nlr,
        "Age": age
    }

    selected_type = cancer_mapping[cancer_type]

    for i in range(1, 17):
        values[f"CancerType{i}"] = 1 if selected_type == i else 0

    inputs = pd.DataFrame(
        [[values[col] for col in model.feature_names_in_]],
        columns=model.feature_names_in_
    )

    probability = float(model.predict_proba(inputs)[0][1])
    non_response_probability = 1 - probability

    threshold = 0.269
    prediction = int(probability >= threshold)

    if prediction == 1:
        result = "Response"
        result_description = (
            "The model classifies this example as belonging to "
            "the response group."
        )
    else:
        result = "Non-response"
        result_description = (
            "The model classifies this example as belonging to "
            "the non-response group."
        )

    return f"""
    <div class="result-panel">

        <div class="result-eyebrow">MODEL OUTPUT</div>

        <h2 class="result-heading">{result}</h2>

        <p class="result-description">{result_description}</p>

        <div class="probability-grid">

            <div class="probability-card">
                <div class="probability-label">Response probability</div>
                <div class="probability-number">{probability * 100:.1f}%</div>
                <div class="bar-background">
                    <div class="bar-response"
                         style="width: {probability * 100:.1f}%"></div>
                </div>
            </div>

            <div class="probability-card">
                <div class="probability-label">Non-response probability</div>
                <div class="probability-number">
                    {non_response_probability * 100:.1f}%
                </div>
                <div class="bar-background">
                    <div class="bar-nonresponse"
                         style="width: {non_response_probability * 100:.1f}%"></div>
                </div>
            </div>

        </div>

        <div class="threshold-note">
            <strong>Decision threshold: 26.9%</strong>
            <p>
                The model uses a threshold of 26.9% to assign its
                classification. A response probability at or above
                this value is classified as Response.
            </p>
        </div>

        <div class="disclaimer">
            <strong>Important:</strong> This is an educational research
            prototype, not a clinically validated tool. Its output must
            not be used to make medical decisions or guide patient treatment.
        </div>

    </div>
    """


# --------------------------------------------------
# 3. VISUAL DESIGN
# --------------------------------------------------

custom_css = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Manrope:wght@400;500;600;700;800&display=swap');

:root {
    --navy: #183344;
    --navy-light: #29495A;
    --sage: #DCE9E2;
    --sage-dark: #638575;
    --cream: #F7F6F1;
    --white: #FFFFFF;
    --text: #263A43;
    --muted: #718087;
    --border: #E1E7E3;
}

.gradio-container {
    font-family: 'DM Sans', sans-serif !important;
    background: var(--cream) !important;
}

h1, h2, h3 {
    font-family: 'Manrope', sans-serif !important;
}

#hero {
    background: var(--navy);
    color: white;
    border-radius: 22px;
    padding: 42px 44px;
    margin: 12px 0 22px 0;
}

#hero .eyebrow {
    color: #B9D2C5;
    font-size: 12px;
    font-weight: 700;
    letter-spacing: 2px;
    text-transform: uppercase;
    margin-bottom: 15px;
}

#hero h1 {
    color: white;
    font-size: 43px;
    line-height: 1.12;
    margin: 0 0 14px 0;
    font-weight: 800;
}

#hero p {
    color: #D6E1E2;
    font-size: 16px;
    line-height: 1.7;
    max-width: 720px;
}

.hero-tag {
    display: inline-block;
    background: #29495A;
    color: #DCE9E2;
    border: 1px solid #55716F;
    padding: 7px 12px;
    border-radius: 30px;
    font-size: 12px;
    margin-top: 12px;
}

.section-heading {
    color: var(--navy);
    font-family: 'Manrope', sans-serif;
    font-size: 25px;
    font-weight: 800;
    margin: 12px 0 8px 0;
}

.section-subtitle {
    color: var(--muted);
    font-size: 14px;
    line-height: 1.7;
    margin-bottom: 18px;
}

.info-card {
    background: white;
    border: 1px solid var(--border);
    border-radius: 15px;
    padding: 20px;
    height: 100%;
}

.info-card h3 {
    color: var(--navy);
    font-size: 16px;
    margin: 0 0 9px 0;
}

.info-card p {
    color: var(--muted);
    font-size: 13px;
    line-height: 1.65;
    margin: 0;
}

.input-description {
    color: #718087;
    font-size: 12px;
    line-height: 1.5;
    margin: -5px 0 12px 0;
}

.result-panel {
    background: white;
    border: 1px solid var(--border);
    border-radius: 18px;
    padding: 25px;
    color: var(--text);
}

.result-eyebrow {
    color: var(--sage-dark);
    font-size: 11px;
    font-weight: 700;
    letter-spacing: 1.8px;
}

.result-heading {
    color: var(--navy);
    font-size: 30px;
    margin: 8px 0;
}

.result-description {
    color: var(--muted);
    font-size: 14px;
    line-height: 1.6;
}

.probability-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 12px;
    margin: 22px 0;
}

.probability-card {
    background: #F7F8F5;
    border-radius: 12px;
    padding: 16px;
}

.probability-label {
    color: var(--muted);
    font-size: 12px;
}

.probability-number {
    color: var(--navy);
    font-size: 25px;
    font-weight: 800;
    margin: 7px 0 12px 0;
}

.bar-background {
    background: #E4EAE6;
    height: 7px;
    border-radius: 10px;
    overflow: hidden;
}

.bar-response {
    background: #638575;
    height: 7px;
    border-radius: 10px;
}

.bar-nonresponse {
    background: #8A9DA2;
    height: 7px;
    border-radius: 10px;
}

.threshold-note {
    background: #F0F4F0;
    border-left: 3px solid #638575;
    border-radius: 8px;
    padding: 14px 16px;
    font-size: 13px;
    line-height: 1.6;
}

.threshold-note p {
    margin: 5px 0 0 0;
    color: var(--muted);
}

.disclaimer {
    background: #F8F3E9;
    border: 1px solid #E9DFCA;
    border-radius: 10px;
    padding: 14px;
    margin-top: 17px;
    color: #695C43;
    font-size: 12px;
    line-height: 1.6;
}

button.primary {
    background: var(--navy) !important;
    border-radius: 10px !important;
}

button.primary:hover {
    background: var(--navy-light) !important;
}

footer {
    display: none !important;
}

@media (max-width: 650px) {
    #hero {
        padding: 28px 23px;
    }
    #hero h1 {
        font-size: 33px;
    }
    .probability-grid {
        grid-template-columns: 1fr;
    }
}
"""


# --------------------------------------------------
# 4. BUILD THE APP
# --------------------------------------------------

with gr.Blocks(
    title="ImmunoPredict | Research Prototype",
    theme=gr.themes.Soft(
        primary_hue="slate",
        secondary_hue="green",
        neutral_hue="slate"
    ),
    css=custom_css
) as demo:

    # HERO / LANDING PAGE

    gr.HTML("""
    <div id="hero">
        <div class="eyebrow">AI · Oncology · Research</div>
        <h1>ImmunoPredict</h1>
        <p>
            Exploring how machine learning can help identify patterns
            associated with immunotherapy response across different
            cancer types.
        </p>
        <span class="hero-tag">An educational machine learning prototype</span>
    </div>
    """)

    with gr.Tabs():

        # ------------------------------------------
        # OVERVIEW TAB
        # ------------------------------------------

        with gr.Tab("Overview"):

            gr.HTML("""
            <div class="section-heading">Understanding the project</div>
            <div class="section-subtitle">
                ImmunoPredict is an interactive demonstration of how
                clinical and biological variables can be used in a
                machine learning classification model.
            </div>
            """)

            with gr.Row():

                gr.HTML("""
                <div class="info-card">
                    <h3>The challenge</h3>
                    <p>
                        Patients can respond differently to immune
                        checkpoint blockade. Identifying patterns
                        associated with response is an important area
                        of ongoing cancer research.
                    </p>
                </div>
                """)

                gr.HTML("""
                <div class="info-card">
                    <h3>The approach</h3>
                    <p>
                        A logistic regression model was trained on
                        published research data, using biological,
                        clinical and cancer-type variables.
                    </p>
                </div>
                """)

                gr.HTML("""
                <div class="info-card">
                    <h3>The objective</h3>
                    <p>
                        To demonstrate how machine learning can
                        explore relationships in research datasets
                        and produce interpretable classification outputs.
                    </p>
                </div>
                """)

            gr.Markdown("""
            ### Explore the model

            The predictor allows you to enter example values for five
            research variables and a cancer type. It then displays the
            model's estimated probabilities for the two outcome classes.

            **This is a research demonstration, not a medical decision tool.**
            """)

        # ------------------------------------------
        # PREDICTOR TAB
        # ------------------------------------------

        with gr.Tab("Explore the model"):

            gr.HTML("""
            <div class="section-heading">Explore a prediction</div>
            <div class="section-subtitle">
                Enter example values below to see how the trained model
                classifies a sample. The descriptions explain each variable.
            </div>
            """)

            with gr.Row():

                with gr.Column():

                    tmb = gr.Number(
                        label="Tumour mutational burden (TMB)",
                        value=32.5
                    )
                    gr.HTML("""
                    <div class="input-description">
                        The number of mutations per megabase of tumour DNA.
                        TMB is studied as a possible indicator of how visible
                        a tumour may be to the immune system.
                    </div>
                    """)

                    therapy = gr.Radio(
                        choices=[
                            ("Yes — previous systemic therapy", 1),
                            ("No — no previous systemic therapy", 0)
                        ],
                        label="Previous systemic therapy",
                        value=1
                    )
                    gr.HTML("""
                    <div class="input-description">
                        Indicates whether the patient had received previous
                        systemic cancer treatment. This can include different
                        types of drug-based treatment.
                    </div>
                    """)

                    albumin = gr.Number(
                        label="Albumin (g/dL)",
                        value=4.3
                    )
                    gr.HTML("""
                    <div class="input-description">
                        Albumin is a protein found in blood plasma. Its level
                        can be associated with general health and inflammation.
                    </div>
                    """)

                with gr.Column():

                    nlr = gr.Number(
                        label="Neutrophil-to-lymphocyte ratio (NLR)",
                        value=1.19
                    )
                    gr.HTML("""
                    <div class="input-description">
                        The ratio of two types of white blood cells.
                        NLR is studied as a possible marker of systemic
                        inflammation and immune activity.
                    </div>
                    """)

                    age = gr.Number(
                        label="Age (years)",
                        value=67.9
                    )
                    gr.HTML("""
                    <div class="input-description">
                        Age of the example patient, expressed in years.
                    </div>
                    """)

                    cancer_type = gr.Dropdown(
                        choices=list(cancer_mapping.keys()),
                        label="Cancer type",
                        value="Bladder"
                    )
                    gr.HTML("""
                    <div class="input-description">
                        Select one of the 16 cancer types represented
                        in the research dataset.
                    </div>
                    """)

            predict_button = gr.Button(
                "Generate model output",
                variant="primary",
                size="lg"
            )

            prediction_output = gr.HTML(
                value="""
                <div class="result-panel">
                    <div class="result-eyebrow">READY TO EXPLORE</div>
                    <h2 class="result-heading">Your results will appear here</h2>
                    <p class="result-description">
                        Enter the example variables and select
                        Generate model output.
                    </p>
                </div>
                """
            )

            predict_button.click(
                fn=predict_response,
                inputs=[
                    tmb,
                    therapy,
                    albumin,
                    nlr,
                    age,
                    cancer_type
                ],
                outputs=prediction_output
            )

        # ------------------------------------------
        # RESEARCH TAB
        # ------------------------------------------

        with gr.Tab("Research & methodology"):

            gr.HTML("""
            <div class="section-heading">The research behind the model</div>
            <div class="section-subtitle">
                ImmunoPredict is based on the Chowell research dataset,
                which brings together clinical and biological information
                from patients treated with immune checkpoint blockade.
            </div>
            """)

            gr.Markdown("""
            ### 1. The dataset

            The project uses the Chowell training and test datasets.
            These contain patient-level variables, cancer-type information
            and recorded treatment-response outcomes.

            The model was trained using the training dataset and evaluated
            against a separate test dataset.

            ### 2. The machine learning approach

            The selected model is **logistic regression**, a supervised
            machine learning algorithm used for classification.

            It estimates the probability of an observation belonging to
            a particular outcome class based on the input variables.

            ### 3. Model performance

            The model achieved the following results on the held-out test set:

            | Evaluation metric | Result |
            |---|---:|
            | Accuracy | 69.7% |
            | ROC-AUC | 0.760 |
            | Precision | 42.7% |
            | Recall | 66.9% |
            | F1 score | 0.521 |

            These metrics describe performance on this particular dataset.
            They do not establish clinical usefulness.

            ### 4. Important limitations

            - The dataset is limited to the patients and treatments represented
              in the original research.
            - The model has not been clinically validated.
            - Its predictions may not generalise to other hospitals,
              populations or treatment settings.
            - Accuracy is lower than the 75.3% majority-class baseline
              on the test set.
            - A probability generated by the model is not a guarantee
              of an individual patient's treatment outcome.

            ### 5. Why this matters

            Machine learning may help researchers investigate complex
            relationships between tumour characteristics, patient
            variables and treatment outcomes.

            However, a model's performance must be assessed carefully
            before it can be considered for real-world clinical use.

            ---

            **Project note:** ImmunoPredict is an independent educational
            prototype developed as part of a TKS research and build project.
            It is not intended for diagnosis, treatment selection or
            individual medical advice.
            """)

    gr.Markdown("""
    ---
    <center>
    <small>ImmunoPredict · Educational research prototype · Not for clinical use</small>
    </center>
    """)


# --------------------------------------------------
# 5. LAUNCH
# --------------------------------------------------

demo.launch(share=True)



/tmp/ipykernel_4184/3923780963.py:367: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://78440e5bab4f374c9d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [17]:
import pandas as pd

source_url = "https://raw.githubusercontent.com/rootchang/LORIS/main/02.Input/AllData.xlsx"

source_file = pd.ExcelFile(source_url)

print("Available sheets:")
print(source_file.sheet_names)

Available sheets:
['Chowell_train', 'Chowell_test', 'MSK1', 'MSK2', 'Kato_panCancer', 'Shim_NSCLC', 'Vanguri_NSCLC', 'Ravi_NSCLC', 'Pradat_panCancer', 'MSK_nonICB']


In [18]:
import pandas as pd

source_url = "https://raw.githubusercontent.com/rootchang/LORIS/main/02.Input/AllData.xlsx"

original_data = pd.read_excel(source_url, sheet_name="Chowell_train")

print("Column names:")
print(original_data.columns.tolist())

print("\nCancer type values:")
print(original_data["CancerType"].value_counts())

Column names:
['SAMPLE_ID', 'Year of ICI treatment start', 'TMB', 'PDL1_TPS(%)', 'Systemic_therapy_history', 'Albumin', 'CancerType', 'CancerType_grouped', 'NLR', 'Age', 'FCNA', 'Drug', 'Sex', 'MSI', 'Stage', 'HLA_LOH', 'HED', 'Platelets', 'HGB', 'BMI', 'PFS_Event', 'PFS_Months', 'OS_Event', 'OS_Months', 'Response', 'CancerType1', 'CancerType2', 'CancerType3', 'CancerType4', 'CancerType5', 'CancerType6', 'CancerType7', 'CancerType8', 'CancerType9', 'CancerType10', 'CancerType11', 'CancerType12', 'CancerType13', 'CancerType14', 'CancerType15', 'CancerType16', 'CancerType17', 'CancerType18']

Cancer type values:
CancerType
NSCLC            331
Melanoma         152
Renal             73
Bladder           65
Head & Neck       48
Gastric           43
Hepatobiliary     36
SCLC              34
Sarcoma           33
Colorectal        28
Endometrial       27
Esophageal        24
Ovarian           23
Mesothelioma      20
Pancreatic        15
Breast            12
Name: count, dtype: int64


In [19]:
# Match each cancer name to its original one-hot encoded column

cancer_mapping = {}

for cancer in original_data["CancerType"].dropna().unique():
    row = original_data[original_data["CancerType"] == cancer].iloc[0]

    matching_columns = [
        f"CancerType{i}"
        for i in range(1, 17)
        if row.get(f"CancerType{i}", 0) == 1
    ]

    cancer_mapping[cancer] = matching_columns

for cancer, columns in cancer_mapping.items():
    print(f"{cancer}: {columns}")

print("\nModel features:")
print([f for f in model.feature_names_in_ if f.startswith("CancerType")])

Colorectal: ['CancerType3']
Melanoma: ['CancerType9']
Hepatobiliary: ['CancerType8']
Gastric: ['CancerType6']
Pancreatic: ['CancerType13']
Renal: ['CancerType14']
Bladder: ['CancerType1']
Sarcoma: ['CancerType15']
SCLC: ['CancerType16']
NSCLC: ['CancerType11']
Endometrial: ['CancerType4']
Head & Neck: ['CancerType7']
Ovarian: ['CancerType12']
Breast: ['CancerType2']
Esophageal: ['CancerType5']
Mesothelioma: ['CancerType10']

Model features:
['CancerType1', 'CancerType2', 'CancerType3', 'CancerType4', 'CancerType5', 'CancerType6', 'CancerType7', 'CancerType8', 'CancerType9', 'CancerType10', 'CancerType11', 'CancerType12', 'CancerType13', 'CancerType14', 'CancerType15', 'CancerType16']


In [22]:
import joblib

joblib.dump(model, "immunopredict_model.joblib")

print("Model exported successfully!")

Model exported successfully!


In [23]:
from google.colab import files

files.download("immunopredict_model.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>